<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../1.%20flask_foundations/3.%20templates_with_jinja2.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 3: Jinja2 Templates</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='5.%20sessions_and_cookies.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 5: Sessions &amp; Cookies →</a>
</div>

# Chapter 4: Handling Forms and User Input — Talking to Your Users

> *"A web app that can't accept input from users is just a brochure."*

## 🎯 The Big Picture

A static website delivers information **one way**. The moment you need a login page, a search box, a comment section, or a signup form — you need **two-way communication**.

Forms are the primary way users talk back to your app. Behind every login button, every search query, every "Post Comment" click — there is an HTML form sending data to your server.

**What you'll learn in this chapter:**
- How browsers send form data (the HTML → HTTP → Flask cycle)
- How Flask's `request` object receives that data
- How **Flask-WTF** makes form handling clean, validated, and secure
- How to display **validation errors** inline next to each field
- How **flash messages** give users feedback after actions

By the end, you'll build a complete login + registration form with CSRF protection, validators, error display, and flash messages.

> ❓ **What is CSRF?** A Cross-Site Request Forgery attack tricks a logged-in user's browser into sending a request to your site without their knowledge. A CSRF token — a secret value embedded in each form — ensures the submission really came from *your* page.

## 🧠 Core Concepts: The "Why"

### The HTML Form → HTTP POST Cycle

When a user fills out a form and clicks Submit, the browser packages the form data and sends it as an **HTTP POST request**:

```json
[User fills form]
       ↓
[Browser sends POST /login  body: username=alice&password=secret]
       ↓
[Flask receives request → route function runs]
       ↓
[Route validates data → decides success or failure]
       ↓
[Flask sends response → redirect or re-render with errors]
```

### The Two HTTP Methods for Forms

| Method | Use case | Data location | Bookmarkable? |
|--------|----------|---------------|---------------|
| `GET`  | Search, filters, navigation | URL: `/search?q=flask` | ✅ Yes |
| `POST` | Login, create, update, delete | Request body (not in URL) | ❌ No |
### Never Trust User Input

This is the **#1 rule of web security**. A user can submit:
- An empty string where you expect a name → **crashes your app**
- A 10MB essay where you expect a 50-char username → **memory issue**
- `<script>alert('hacked')</script>` as their "name" → **XSS attack**
- `'; DROP TABLE users; --` → **SQL injection**

**Validation** = check that input meets your rules before using it.
**Sanitization** = escape/clean the data before storing or displaying it.

Flask-WTF handles validation. Jinja2's auto-escaping handles XSS. SQLAlchemy's parameterized queries handle SQL injection.

> ❓ **Why use an ORM instead of raw SQL?** An ORM (Object-Relational Mapper) lets you work with database rows as Python objects, so you write Python instead of SQL strings. It also prevents SQL injection by default and makes switching databases straightforward later.

## ⚡ Syntax & First Usage: The `request` Object

Flask's `request` object gives you everything the browser sent — form data, query params, JSON, files, and headers.

In [ ]:
attrs = [
    ("request.method",       "'GET', 'POST', 'PUT', 'DELETE', 'PATCH'"),
    ("request.form",         "ImmutableMultiDict — POST body form fields"),
    ("request.args",         "ImmutableMultiDict — URL query params (?key=val)"),
    ("request.get_json()",   "Parsed JSON body (None if not JSON content-type)"),
    ("request.files",        "ImmutableMultiDict of uploaded FileStorage objects"),
    ("request.headers",      "HTTP headers sent by the browser"),
    ("request.cookies",      "Cookies sent by the browser"),
    ("request.host",         "'localhost:5000'"),
    ("request.url",          "'http://localhost:5000/login'"),
    ("request.remote_addr",  "Client IP address"),
]
print("Flask request object — key attributes:")
print()
for attr, desc in attrs:
    print(f"  {attr:<28}  ->  {desc}")


### Safe vs Unsafe Field Access — A Critical Distinction

`request.form['key']` raises a `KeyError` if the field is missing; `request.form.get('key')` returns `None`. Choosing the wrong accessor is one of the most common sources of 400 errors in Flask form handling.

In [ ]:
from werkzeug.datastructures import ImmutableMultiDict

# Simulate a POST where the user only submitted 'username' (forgot 'email')
form_data = ImmutableMultiDict([('username', 'alice')])

print("=== SAFE: .get() — always use this ===")
username = form_data.get('username')
email    = form_data.get('email')            # missing -> None, no crash
role     = form_data.get('role', 'viewer')   # missing -> 'viewer' default
print(f"  username : {username!r}")
print(f"  email    : {email!r}")
print(f"  role     : {role!r}")

print()
print("=== UNSAFE: direct key access ===")
try:
    email = form_data['email']
except KeyError as e:
    print(f"  form_data['email'] -> KeyError: {e}")
    print("  This would show users a 500 Internal Server Error page!")

print()
print("RULE:")
print("  .get('field')         safe — returns None if key missing")
print("  .get('field','val')   safe — returns default if key missing")
print("  ['field']             unsafe — KeyError if key missing")


## 🔬 Deep Dive: Building a Login Form Two Ways

### Approach 1: Manual Forms (No Extensions)

Let's build a complete login form manually first — so you understand exactly what Flask-WTF saves you from.

In [ ]:
# MANUAL APPROACH — app.py (what you write WITHOUT Flask-WTF)
manual = '''
from flask import Flask, request, render_template, redirect, url_for

app = Flask(__name__)
USERS = {"alice": "password123", "bob": "secret456"}   # demo only

@app.route("/login", methods=["GET", "POST"])
def login():
    error = None
    username_value = ""    # re-populate field on error

    if request.method == "POST":
        username = request.form.get("username", "").strip()
        password = request.form.get("password", "")
        username_value = username

        if not username:
            error = "Username is required."
        elif len(username) < 2:
            error = "Username must be at least 2 characters."
        elif len(username) > 50:
            error = "Username must be at most 50 characters."
        elif not password:
            error = "Password is required."
        elif len(password) < 8:
            error = "Password must be at least 8 characters."
        elif username not in USERS:
            error = "Invalid username or password."
        elif USERS[username] != password:
            error = "Invalid username or password."
        else:
            return redirect(url_for("dashboard"))

    return render_template("login.html", error=error, username=username_value)
'''
print(manual)
print()
print("Problems with the manual approach:")
for p in [
    "No CSRF protection — vulnerable to cross-site request forgery",
    "Validation logic scattered inside the route function",
    "Must manually track + re-populate field values on error",
    "No reusable validators — copy-paste for every form",
    "Single global error message (not per-field)",
    "5-field form = ~100 lines of validation code",
]:
    print(f"  x  {p}")


### Approach 2: Flask-WTF — The Right Way

**Flask-WTF** wraps the powerful **WTForms** library and adds Flask integration. You declare your form as a Python class — fields, labels, validators. The library handles everything else.

**You get for free:**
- 🛡️ Automatic CSRF protection (just `{{ form.hidden_tag() }}` in template)
- ✅ Declarative validators defined once in the Form class
- 🔄 Auto-repopulation of all fields on validation failure
- 📋 Per-field error lists (`form.username.errors`)
- 🔧 Custom validators with `validate_fieldname()` methods

In [ ]:
# pip install flask-wtf email-validator

# Form class (forms.py)
form_code = '''
from flask_wtf import FlaskForm
from wtforms import (StringField, PasswordField, SubmitField,
                     BooleanField, TextAreaField, SelectField)
from wtforms.validators import (DataRequired, Length, Email,
                                EqualTo, Regexp, ValidationError)

class LoginForm(FlaskForm):
    # StringField  -> <input type="text">
    # DataRequired -> fails if field is empty or only whitespace
    # Length       -> enforces min/max character count
    username = StringField(
        "Username",
        validators=[
            DataRequired(message="Username is required."),
            Length(min=2, max=50, message="Username must be 2-50 characters.")
        ]
    )

    # PasswordField -> <input type="password"> (never repopulated on error)
    password = PasswordField(
        "Password",
        validators=[
            DataRequired(message="Password is required."),
            Length(min=8, message="Password must be at least 8 characters.")
        ]
    )

    remember_me = BooleanField("Remember Me")   # <input type="checkbox">
    submit      = SubmitField("Login")          # <input type="submit">


class RegistrationForm(FlaskForm):
    username = StringField("Username", validators=[
        DataRequired(),
        Length(min=2, max=50),
        Regexp(r"^[A-Za-z0-9_]+$",
               message="Letters, numbers, and underscores only.")
    ])

    email = StringField("Email", validators=[    # pip install email-validator
        DataRequired(),
        Email(message="Please enter a valid email address.")
    ])

    password = PasswordField("Password", validators=[
        DataRequired(), Length(min=8)
    ])

    # EqualTo: this field value must match another field
    confirm_password = PasswordField("Confirm Password", validators=[
        DataRequired(), EqualTo("password", message="Passwords must match.")
    ])

    bio    = TextAreaField("Bio (optional)", validators=[Length(max=500)])
    submit = SubmitField("Create Account")

    # Custom validator: named validate_<fieldname>
    # Called automatically AFTER built-in validators pass
    def validate_username(self, username):
        reserved = ["admin", "root", "system", "superuser"]
        if username.data.lower() in reserved:
            raise ValidationError("That username is reserved.")
'''
print(form_code)


In [ ]:
# Route using Flask-WTF
route_code = '''
from flask import Flask, render_template, redirect, url_for, flash
from forms import LoginForm, RegistrationForm

app = Flask(__name__)
app.config["SECRET_KEY"] = "your-strong-random-secret-key"   # REQUIRED

@app.route("/login", methods=["GET", "POST"])
def login():
    form = LoginForm()    # Always create a fresh instance

    # validate_on_submit() returns True ONLY when ALL of:
    #   1. Request method is POST (not GET)
    #   2. All field validators pass
    #   3. CSRF token is present and valid
    if form.validate_on_submit():
        username = form.username.data        # cleaned, stripped value
        password = form.password.data
        remember = form.remember_me.data     # True or False

        if username == "alice" and password == "password123":
            flash(f"Welcome back, {username}!", "success")
            return redirect(url_for("dashboard"))
        else:
            flash("Invalid username or password.", "danger")

    # GET request OR failed validation: render the form
    # form object already carries validation errors
    return render_template("login.html", form=form)


@app.route("/register", methods=["GET", "POST"])
def register():
    form = RegistrationForm()
    if form.validate_on_submit():
        flash(f"Account created for {form.username.data}!", "success")
        return redirect(url_for("login"))
    return render_template("register.html", form=form)
'''
print(route_code)


In [ ]:
# Template: login.html with Flask-WTF
template = '''
<!DOCTYPE html>
<html>
<head><title>Login</title></head>
<body>
<h2>Login</h2>

<!-- Flash messages block (put in base.html so ALL pages get it) -->
{%- with messages = get_flashed_messages(with_categories=true) %}
  {%- if messages %}
    {%- for category, message in messages %}
      <div style="padding:10px; margin:8px 0; border-radius:4px;">
        {{ message }}
      </div>
    {%- endfor %}
  {%- endif %}
{%- endwith %}

<form method="POST" action="">

  <!-- This ONE line renders the hidden CSRF token field -->
  {{ form.hidden_tag() }}

  <!-- Username: label + input widget + per-field errors -->
  <div style="margin-bottom:12px;">
    {{ form.username.label }}<br>
    {{ form.username(size=32, placeholder="Enter username") }}<br>
    {%- for error in form.username.errors %}
      <span style="color:red; font-size:0.85em;">{{ error }}</span><br>
    {%- endfor %}
  </div>

  <!-- Password -->
  <div style="margin-bottom:12px;">
    {{ form.password.label }}<br>
    {{ form.password(size=32) }}<br>
    {%- for error in form.password.errors %}
      <span style="color:red; font-size:0.85em;">{{ error }}</span><br>
    {%- endfor %}
  </div>

  <!-- Checkbox: widget FIRST, then label -->
  <div style="margin-bottom:12px;">
    {{ form.remember_me() }} {{ form.remember_me.label }}
  </div>

  {{ form.submit(style="padding:8px 20px; background:#007bff; color:white; border:none; border-radius:4px;") }}
</form>
</body>
</html>
'''
print(template)


### ⚖️ Manual vs Flask-WTF — Side-by-Side Comparison

Manual forms require less setup but put validation and CSRF protection entirely on you. Flask-WTF adds a thin layer of boilerplate in exchange for automatic CSRF tokens, clean field definitions, and reusable validators.

In [ ]:
lines = [
    "+--------------------------------+----------------------+----------------------------------+",
    "| Feature                        | Manual               | Flask-WTF                        |",
    "+--------------------------------+----------------------+----------------------------------+",
    "| CSRF Protection                | None (vulnerable!)   | Automatic (one template line)    |",
    "| Validation location            | Inside route fn      | Centralized in Form class        |",
    "| Field repopulation on error    | Manual, tedious      | Automatic                        |",
    "| Reusable validators            | Copy-paste           | Import and share across forms    |",
    "| Per-field error messages       | One global error     | form.field.errors list           |",
    "| Data type coercion             | Manual str->int etc. | Field type handles it            |",
    "| Custom validator syntax        | if/else in route     | validate_fieldname() method      |",
    "| Email validation               | Manual regex         | Email() validator                |",
    "| Password confirmation match    | Manual compare       | EqualTo() validator              |",
    "| Lines of code (login form)     | ~80 lines            | ~25 lines                        |",
    "+--------------------------------+----------------------+----------------------------------+",
    "",
    "Verdict: Always use Flask-WTF for real projects.",
]
for line in lines:
    print(line)


### All WTForms Field Types and Validators at a Glance

WTForms ships with field types for every common HTML input and a library of validators you can chain together. The table below is your quick reference for picking the right field and validator combination.

In [ ]:
field_lines = [
    "FIELD TYPES                        RENDERS AS",
    "-----------                        ----------",
    "StringField('Label')             -> <input type='text'>",
    "PasswordField('Label')           -> <input type='password'>  (never repopulated)",
    "TextAreaField('Label')           -> <textarea>",
    "IntegerField('Label')            -> <input type='number'>    (int coercion)",
    "FloatField('Label')              -> float coercion",
    "BooleanField('Label')            -> <input type='checkbox'>",
    "SelectField('L', choices=[...])  -> <select>",
    "SelectMultipleField(...)         -> <select multiple>",
    "RadioField('Label', choices=[])  -> radio button group",
    "DateField('Label')               -> date input with date coercion",
    "FileField('Label')               -> <input type='file'>",
    "HiddenField('Label')             -> <input type='hidden'>",
    "SubmitField('Label')             -> <input type='submit'>",
    "",
    "VALIDATORS                         DESCRIPTION",
    "----------                         -----------",
    "DataRequired()                   -> field must not be empty/whitespace",
    "Optional()                       -> skip other validators if field is empty",
    "Length(min=2, max=50)            -> character count range",
    "NumberRange(min=0, max=100)      -> numeric value range",
    "Email()                          -> valid email format",
    "URL()                            -> valid URL format",
    "Regexp(r'^[A-Za-z]+$')          -> must match regex pattern",
    "EqualTo('field_name')            -> must equal another field's value",
    "AnyOf(['a', 'b', 'c'])           -> value must be in the list",
    "NoneOf(['admin', 'root'])        -> value must NOT be in the list",
]
for line in field_lines:
    print(line)


### Flash Messages — One-Time Feedback That Survives Redirects

After a form submission you typically redirect rather than render directly (Post/Redirect/Get pattern). Flash messages let you carry a one-time status string — "Saved!", "Invalid password" — across that redirect so the user sees feedback on the next page.

In [ ]:
flash_code = '''
from flask import flash

# In routes: flash before redirecting
@app.route("/delete/<int:post_id>", methods=["POST"])
def delete_post(post_id):
    # ... delete from database ...
    flash("Post deleted successfully.", "success")
    return redirect(url_for("blog_list"))   # message survives this redirect!

# Multiple flashes can be stacked:
flash("Logged in.", "success")
flash("You have 3 unread messages.", "info")

# Common categories (just strings for CSS class names):
#   "success"  -> green  (operation succeeded)
#   "danger"   -> red    (error or invalid input)
#   "warning"  -> yellow (caution, heads-up)
#   "info"     -> blue   (neutral information)

# In base.html template (shows on ALL pages):
# {%- with messages = get_flashed_messages(with_categories=true) %}
#   {%- if messages %}
#     {%- for category, message in messages %}
#       <div class="alert alert-{{ category }}">
#         {{ message }}
#         <button onclick="this.parentElement.remove()">x</button>
#       </div>
#     {%- endfor %}
#   {%- endif %}
# {%- endwith %}
'''
print(flash_code)
print()
print("Key facts about flash messages:")
for f in [
    "Stored in the SESSION (requires SECRET_KEY)",
    "Consumed ONCE — get_flashed_messages() clears them from session",
    "Survive exactly one redirect, then gone",
    "Category is just a string you use for CSS class names",
    "Multiple flashes can be stacked before a single redirect",
]:
    print(f"  - {f}")


## 🧪 What If? Experimentation

### What If 1: A User Submits an Empty Required Field?

When a form is submitted without filling in a required field, validation should catch it before any processing occurs. Here's how WTForms handles this and what response the user should see:


In [ ]:
# Simulating WTForms validation directly (no Flask server needed)
from wtforms import Form, StringField, PasswordField
from wtforms.validators import DataRequired, Length
from werkzeug.datastructures import ImmutableMultiDict

class SimpleLoginForm(Form):
    username = StringField('Username', validators=[
        DataRequired(message="Username is required."),
        Length(min=2, max=50, message="Must be 2-50 characters.")
    ])
    password = PasswordField('Password', validators=[
        DataRequired(message="Password is required."),
        Length(min=8, message="Must be at least 8 characters.")
    ])

print("=== Test 1: Completely empty submission ===")
f1 = SimpleLoginForm(ImmutableMultiDict([]))
print(f"  Valid: {f1.validate()}")
print(f"  username errors: {f1.username.errors}")
print(f"  password errors: {f1.password.errors}")

print()
print("=== Test 2: Username too short (1 char) ===")
f2 = SimpleLoginForm(ImmutableMultiDict([('username', 'a'), ('password', 'mypassword')]))
print(f"  Valid: {f2.validate()}")
print(f"  username errors: {f2.username.errors}")

print()
print("=== Test 3: Password too short ===")
f3 = SimpleLoginForm(ImmutableMultiDict([('username', 'alice'), ('password', '123')]))
print(f"  Valid: {f3.validate()}")
print(f"  password errors: {f3.password.errors}")

print()
print("=== Test 4: Everything valid ===")
f4 = SimpleLoginForm(ImmutableMultiDict([('username', 'alice'), ('password', 'password123')]))
print(f"  Valid: {f4.validate()}")
print(f"  username data: {f4.username.data!r}")

print()
print("On validation failure:")
print("  - validate_on_submit() returns False")
print("  - Route re-renders the template (no redirect)")
print("  - form.field.errors is populated with messages")
print("  - Template loops: {%- for e in form.username.errors %}")


### What If 2: The CSRF Token is Missing or Invalid?

Flask-WTF rejects any POST without a valid CSRF token with a 400 error. This usually means the hidden `{{ form.hidden_tag() }}` is absent from your template, the form was submitted from a different origin, or the session expired.

In [ ]:
print('''
SCENARIO: POST request arrives without a valid CSRF token

What Flask-WTF does:
  1. Intercepts the request BEFORE your route function runs
  2. Checks for "csrf_token" in the POST body (or X-CSRFToken header)
  3. Missing or invalid token -> raises CSRFError -> HTTP 400 Bad Request
  4. Your route function is never called

WHY CSRF ATTACKS ARE DANGEROUS (without protection):
  1. You are logged into mybank.com (browser holds your auth cookie)
  2. You visit evil.com in another tab
  3. evil.com page silently submits a form to mybank.com/transfer
  4. Your browser attaches your mybank.com cookie automatically!
  5. Bank sees a valid session and executes the transfer

HOW THE CSRF TOKEN STOPS THIS:
  - {{ form.hidden_tag() }} renders a cryptographically signed secret token
  - Token is tied to YOUR session and has an expiry time
  - evil.com cannot read the token (same-origin policy blocks it)
  - Without valid token -> request rejected with 400 Bad Request

HANDLE THE ERROR GRACEFULLY IN PRODUCTION:

  from flask_wtf.csrf import CSRFError

  @app.errorhandler(CSRFError)
  def handle_csrf_error(e):
      flash("Your form session expired. Please try again.", "warning")
      return redirect(request.referrer or url_for("index"))
''')


### What If 3: Accessing `request.form` on a GET Request?

When your route handles both `GET` and `POST`, understanding what `request.form` contains on each matters.

On a `GET` request (page load), there is **no form body** — the browser is just asking for the page. `request.form` is an empty `ImmutableMultiDict`. This is completely normal and expected.

The correct pattern:

```python
@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        # POST: browser submitted the form; data is in request.form
        username = request.form.get('username', '')
        password = request.form.get('password', '')
        # ... validate and process
    # GET (or failed POST): show the empty form
    return render_template('login.html')
```

With Flask-WTF this is even cleaner:

```python
@app.route('/login', methods=['GET', 'POST'])
def login():
    form = LoginForm()
    if form.validate_on_submit():   # False on GET, True on valid POST
        # ... process login
    return render_template('login.html', form=form)
```

`form.validate_on_submit()` returns `False` on GET without raising any errors, so the blank form is shown. On POST with valid data it returns `True`. On POST with invalid data it returns `False` and populates `form.errors`.

In [ ]:
from werkzeug.datastructures import ImmutableMultiDict

# On a GET request, there is no form body.
# request.form is an empty ImmutableMultiDict.
get_form = ImmutableMultiDict()    # simulates request.form on GET

print("=== request.form on a GET request ===")
print(f"  Type: {type(get_form).__name__}")
print(f"  Contents: {dict(get_form)}")

print()
print("=== Accessing fields ===")
val = get_form.get("username")
print(f"  .get('username') -> {val!r}  (safe, returns None)")

try:
    val2 = get_form["username"]
except KeyError as e:
    print(f"  ['username']     -> KeyError: {e}  (crashes!)")

print()
print("=== validate_on_submit() handles GET correctly ===")
print("  On GET:  returns False immediately")
print("           No validation runs. Empty form rendered cleanly.")
print()
print("  On POST: runs all validators + checks CSRF")
print("           Returns True only if everything passes.")
print()
print("  This is why you never need to write:")
print("    if request.method == 'POST':      # old manual pattern")
print("  With Flask-WTF, just write:")
print("    if form.validate_on_submit():     # handles everything")


## 🚀 Real-World Project Link

Our **Blog** application uses Flask-WTF for every user-facing form: the **registration form** (username, email, password with confirmation + custom uniqueness validator querying the database), the **login form** (with Remember Me checkbox), the **post creation form** (title, content TextArea, tags), and the **comment form**. Every form gets automatic CSRF protection, inline per-field validation errors, and flash message feedback — without repetitive validation code in routes.

### Custom Validators — Business-Logic Validation

WTForms' built-in validators (`DataRequired`, `Email`, `Length`) cover standard cases. For application-specific rules — "is this username already taken?", "is the end date after the start date?" — you write **custom validators**.

**Method 1: `validate_<fieldname>()` on the form class (per-field):**

```python
class RegistrationForm(FlaskForm):
    username = StringField('Username', validators=[DataRequired(), Length(2, 50)])
    email    = EmailField('Email',    validators=[DataRequired(), Email()])

    def validate_username(self, field):
        """Custom validator: username must not already exist in the database."""
        if User.query.filter_by(username=field.data).first():
            raise ValidationError(
                f'The username "{field.data}" is already taken. Please choose another.'
            )

    def validate_email(self, field):
        """Custom validator: email must be unique."""
        if User.query.filter_by(email=field.data.lower()).first():
            raise ValidationError('An account with this email already exists.')
```

WTForms automatically calls `validate_<fieldname>()` methods during validation. Raise `ValidationError` with a message; the error appears in `form.<fieldname>.errors` and in your template next to the field.

**Method 2: Standalone validator function (reusable across forms):**

```python
from wtforms.validators import ValidationError

def strong_password(form, field):
    """Reusable: password must have uppercase, digit, and special char."""
    import re
    p = field.data
    if not re.search(r'[A-Z]', p): raise ValidationError('Must have an uppercase letter.')
    if not re.search(r'\d', p):    raise ValidationError('Must have a digit.')
    if not re.search(r'[!@#$%]',p):raise ValidationError('Must have !@#$% special char.')

class RegistrationForm(FlaskForm):
    password = PasswordField('Password', validators=[
        DataRequired(), Length(min=8), strong_password   # reuse it here
    ])

class ChangePasswordForm(FlaskForm):
    new_password = PasswordField('New Password', validators=[
        DataRequired(), Length(min=8), strong_password   # and here
    ])
```

**Cross-field validation** (comparing two fields):

```python
class EventForm(FlaskForm):
    start_date = DateField('Start', validators=[DataRequired()])
    end_date   = DateField('End',   validators=[DataRequired()])

    def validate_end_date(self, field):
        if field.data <= self.start_date.data:
            raise ValidationError('End date must be after start date.')
```

## 📋 Chapter Summary & Cheat Sheet

The key patterns from this chapter — manual form parsing, Flask-WTF validation, CSRF protection, and flash messages — are summarised below for quick reference.

In [ ]:
lines = [
    "# ============================================================",
    "#      CHAPTER 4 CHEAT SHEET — FLASK FORMS & USER INPUT",
    "# ============================================================",
    "",
    "# SETUP",
    "# pip install flask-wtf email-validator",
    'app.config["SECRET_KEY"] = "your-strong-secret"   # REQUIRED',
    "",
    "# FORM CLASS (forms.py)",
    "from flask_wtf import FlaskForm",
    "from wtforms import StringField, PasswordField, BooleanField, SubmitField",
    "from wtforms.validators import DataRequired, Length, Email, EqualTo",
    "",
    "class LoginForm(FlaskForm):",
    '    username = StringField("Username", validators=[DataRequired(), Length(2,50)])',
    '    password = PasswordField("Password", validators=[DataRequired(), Length(8)])',
    '    remember = BooleanField("Remember Me")',
    '    submit   = SubmitField("Login")',
    "",
    "# Custom validator — runs automatically after built-in validators",
    "def validate_username(self, username):",
    '    if username.data == "banned":',
    '        raise ValidationError("That username is not allowed.")',
    "",
    "# ROUTE",
    '@app.route("/login", methods=["GET", "POST"])',
    "def login():",
    "    form = LoginForm()",
    '    if form.validate_on_submit():      # POST + all valid + CSRF OK',
    "        data = form.username.data      # cleaned value",
    '        flash("Welcome!", "success")',
    '        return redirect(url_for("dashboard"))',
    '    return render_template("login.html", form=form)',
    "",
    "# TEMPLATE",
    "# <form method='POST'>",
    "#   {{ form.hidden_tag() }}               CSRF token",
    "#   {{ form.username.label }}             <label>",
    "#   {{ form.username(placeholder='...') }} <input> + HTML attrs",
    "#   {%- for e in form.username.errors %}",
    "#     <span style='color:red'>{{ e }}</span>",
    "#   {%- endfor %}",
    "#   {{ form.submit() }}",
    "# </form>",
    "",
    "# FLASH MESSAGES",
    'flash("Done!", "success")    # success | danger | warning | info',
    "# {%- with messages = get_flashed_messages(with_categories=true) %}",
    "#   {%- for cat, msg in messages %}",
    "#     <div class='alert-{{ cat }}'>{{ msg }}</div>",
    "",
    "# REQUEST OBJECT",
    "request.method              # 'GET' or 'POST'",
    "request.form.get('field')   # safe: None if missing",
    "request.form['field']       # unsafe: KeyError if missing",
    "request.args.get('q')       # GET param from ?q=value",
    "request.get_json()          # parsed JSON body (or None)",
]
for line in lines:
    print(line)


### SelectField and Multi-Choice Fields

`SelectField` renders an HTML `<select>` and validates that the submitted value is one of the allowed choices. `SelectMultipleField` works the same way but accepts a list of selected values.

In [ ]:
# SelectField — dropdowns and radio buttons
from wtforms import Form, SelectField, SelectMultipleField, RadioField
from werkzeug.datastructures import ImmutableMultiDict

class PostForm(Form):
    # SelectField: choices is list of (value, label) tuples
    category = SelectField(
        "Category",
        choices=[
            ("tech",     "Technology"),
            ("science",  "Science"),
            ("art",      "Arts & Culture"),
            ("politics", "Politics"),
        ]
    )

    # SelectMultipleField: user can pick several
    tags = SelectMultipleField(
        "Tags",
        choices=[("python","Python"), ("flask","Flask"),
                 ("web","Web Dev"), ("db","Databases")]
    )

    # RadioField: exclusive choices rendered as radio buttons
    visibility = RadioField(
        "Visibility",
        choices=[("public","Public"), ("private","Private"),
                 ("draft","Draft")],
        default="public"
    )

# Test it
form = PostForm(ImmutableMultiDict([
    ('category', 'tech'),
    ('tags', 'python'),
    ('tags', 'flask'),
    ('visibility', 'public'),
]))
print(f"category:   {form.category.data!r}")
print(f"tags:       {form.tags.data}")
print(f"visibility: {form.visibility.data!r}")


### File Upload Handling

File uploads require `enctype="multipart/form-data"` on the HTML form and `request.files` (not `request.form`) on the server. Always validate the filename with `secure_filename()` and check the extension before saving.

In [ ]:
# File uploads with Flask-WTF
file_upload_code = '''
from flask_wtf import FlaskForm
from flask_wtf.file import FileField, FileRequired, FileAllowed
from wtforms import SubmitField

class ProfilePhotoForm(FlaskForm):
    # FileField with validators:
    # FileRequired()             -> a file must be selected
    # FileAllowed([...])         -> only these extensions are allowed
    photo = FileField(
        "Profile Photo",
        validators=[
            FileRequired(message="Please select a photo."),
            FileAllowed(["jpg", "jpeg", "png", "gif"],
                        message="Only image files are allowed.")
        ]
    )
    submit = SubmitField("Upload")


# Route handling file upload:
import os
from werkzeug.utils import secure_filename

@app.route("/upload-photo", methods=["GET", "POST"])
@login_required
def upload_photo():
    form = ProfilePhotoForm()
    if form.validate_on_submit():
        f = form.photo.data                  # FileStorage object
        filename = secure_filename(f.filename)  # sanitize filename!
        save_path = os.path.join(app.config["UPLOAD_FOLDER"], filename)
        f.save(save_path)
        flash(f"Photo uploaded: {filename}", "success")
        return redirect(url_for("profile"))
    return render_template("upload.html", form=form)

# Config:
# app.config["UPLOAD_FOLDER"] = "static/uploads"
# app.config["MAX_CONTENT_LENGTH"] = 16 * 1024 * 1024  # 16MB limit
'''
print(file_upload_code)

print()
print("Security rules for file uploads:")
for rule in [
    "ALWAYS use secure_filename() — prevents path traversal attacks",
    "Validate file extensions with FileAllowed()",
    "Set MAX_CONTENT_LENGTH to limit file size",
    "Store uploads OUTSIDE the web root when possible",
    "Consider using cloud storage (S3, GCS) for production",
]:
    print(f"  - {rule}")


### The POST/Redirect/GET Pattern

This is the most important web development pattern for form handling. Without it, refreshing the page after a form submission re-submits the form.

In [ ]:
prg_pattern = '''
# POST/Redirect/GET (PRG) Pattern
#
# PROBLEM without PRG:
#   1. User submits form (POST /create-post)
#   2. Server processes + renders success page
#   3. User presses F5 (refresh)
#   4. Browser asks "Resend POST data?" -> re-submits form
#   5. Duplicate post created!
#
# SOLUTION: Always redirect after a successful POST
#
# WITHOUT PRG (bad):
@app.route("/create-post", methods=["GET", "POST"])
def create_post_bad():
    form = PostForm()
    if form.validate_on_submit():
        post = Post(title=form.title.data, ...)
        db.session.add(post)
        db.session.commit()
        return render_template("success.html", post=post)  # BAD: renders directly
        # Refresh now re-submits the POST!

# WITH PRG (correct):
@app.route("/create-post", methods=["GET", "POST"])
def create_post_good():
    form = PostForm()
    if form.validate_on_submit():
        post = Post(title=form.title.data, ...)
        db.session.add(post)
        db.session.commit()
        flash(f"Post created: {post.title}", "success")
        return redirect(url_for("view_post", post_id=post.id))  # REDIRECT!
        # Now refresh loads GET /view-post/123 (safe, idempotent)
    return render_template("create_post.html", form=form)
'''
print(prg_pattern)

print()
print("PRG Rule: After ANY successful form POST:")
print("  db.session.commit()  -> flash()  -> redirect()")
print("NEVER render_template() as the response to a successful POST.")


### Form Customisation: HTML Attributes and Macros

WTForms fields accept `render_kw` to inject arbitrary HTML attributes (`class`, `placeholder`, `id`, etc.) into the rendered input element. This lets you apply CSS frameworks (Bootstrap, Tailwind) without modifying Python code.

```python
class LoginForm(FlaskForm):
    username = StringField('Username', validators=[DataRequired()], render_kw={
        "class":       "form-control",
        "placeholder": "Enter your username",
        "autofocus":   True,
    })
    password = PasswordField('Password', validators=[DataRequired()], render_kw={
        "class":       "form-control",
        "placeholder": "Enter your password",
    })
```

**You can also pass HTML attributes at render time in the template:**

```html
{# Override or add attributes per-instance #}
{{ form.username(class="form-control form-control-lg", placeholder="Username") }}
```

**Jinja2 macro for DRY form rendering:**

Once you have more than 2-3 forms, copy-pasting the `<div class="mb-3">` wrapper and error list for every field becomes tedious. A macro lets you render any WTForms field with your design system's markup in a single call:

```html
{% macro render_field(field, class='form-control') %}
  <div class="mb-3">
    {{ field.label(class='form-label') }}
    {{ field(class=class, **kwargs) }}
    {% for error in field.errors %}
      <div class="invalid-feedback d-block">{{ error }}</div>
    {% endfor %}
  </div>
{% endmacro %}

{# Usage -- clean and consistent across all templates #}
{{ render_field(form.username) }}
{{ render_field(form.email) }}
{{ render_field(form.password) }}
{{ render_field(form.submit, class='btn btn-primary') }}
```

In [ ]:
# Passing HTML attributes to form widgets
template_attrs = '''
<!-- Pass any HTML attribute to the widget as keyword arguments -->
{{ form.username(
    class="form-control",
    placeholder="Enter username",
    autocomplete="username",
    autofocus=True,
    id="username-field"
) }}

<!-- Renders as: -->
<!-- <input class="form-control" placeholder="Enter username"
            autocomplete="username" autofocus id="username-field"
            id="username" name="username" type="text" value=""> -->

<!-- Jinja2 macro for DRY form rendering (define once, use everywhere) -->
{% macro render_field(field) %}
  <div class="form-group mb-3">
    {{ field.label(class="form-label fw-bold") }}
    {{ field(class="form-control" + (" is-invalid" if field.errors else "")) }}
    {% for error in field.errors %}
      <div class="invalid-feedback">{{ error }}</div>
    {% endfor %}
  </div>
{% endmacro %}

<!-- Usage in any template: -->
{% from "macros.html" import render_field %}
<form method="POST">
  {{ form.hidden_tag() }}
  {{ render_field(form.username) }}
  {{ render_field(form.email) }}
  {{ render_field(form.password) }}
  {{ form.submit(class="btn btn-primary") }}
</form>
'''
print(template_attrs)


## 💪 Practice Prompts

**Challenge 1 — Contact Form:**
Build a `/contact` route with a `ContactForm` containing: `name` (required, 2–50 chars), `email` (required, valid email), `subject` (required, 5–100 chars), and `message` (TextAreaField, required, 20–1000 chars). On successful submit, flash "Message sent! We'll reply within 24 hours." and redirect to home. Display all validation errors inline next to each field.

**Challenge 2 — Registration with Custom Validators:**
Create a `RegistrationForm` with username, email, password, and confirm_password. Add `validate_username` that raises `ValidationError('Username already taken.')` if the username exists in a module-level dict. Add `validate_password` that requires at least one digit and one uppercase letter. Test all validation branches with `app.test_client()`.

**Challenge 3 — Search with Query Parameters:**
Build a `/search` route that accepts `q` (search query) and `page` (integer, default 1) via query string (`request.args`). Return a JSON list of matching results from a hard-coded dataset. Add a `GET` form that submits to the same route. Show how `url_for('search', q='flask', page=2)` generates the correct URL.

**Challenge 4 — Complete PRG with Flash Messages:**
Implement a "Tasks" list:
- `GET /tasks` → list all tasks
- `POST /tasks` → add a task, flash success, redirect 303 to `GET /tasks`
- `POST /tasks/<int:id>/delete` → delete task, flash "Task deleted.", redirect 303 to `GET /tasks`

Verify with `app.test_client()` that:
- After `POST /tasks`, calling `GET /tasks` shows the new task
- A second `GET /tasks` (simulating F5 refresh) does NOT duplicate the task
- The flash message appears once and disappears on the next request

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../1.%20flask_foundations/3.%20templates_with_jinja2.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 3: Jinja2 Templates</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='5.%20sessions_and_cookies.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 5: Sessions &amp; Cookies →</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../1. flask_foundations/3. templates_with_jinja2.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='5. sessions_and_cookies.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>